# REST vs GraphQL — comparación con código

Tutorial de la **Sesión 2** de MLOps II. Servimos **la misma información** (modelos y sus métricas) por **REST** y por **GraphQL** en una misma app FastAPI, y comparamos qué pasa al armar una vista típica: *el nombre y el AUC de todos los modelos*.

Vamos a ver, con números, dos problemas clásicos de REST:
* **Over-fetching:** el endpoint devuelve objetos completos aunque solo querías dos campos.
* **Under-fetching / N+1:** para juntar datos relacionados hacen falta varias llamadas.

Y cómo **GraphQL** los resuelve pidiendo *exactamente* lo necesario en **una** consulta.

## 1. Requisitos

```bash
uv pip install strawberry-graphql fastapi "uvicorn[standard]" requests
```
Usa el kernel del entorno de uv.

In [ ]:
# Si falta algo en el kernel, descomenta:
# !uv pip install strawberry-graphql fastapi "uvicorn[standard]" requests

## 2. La app: la misma data por REST y por GraphQL

Definimos unos modelos con sus métricas y montamos, en un mismo FastAPI:
* **REST:** `GET /rest/models` (lista, **sin** métricas) y `GET /rest/models/{id}/metrics` (las métricas de uno).
* **GraphQL:** un endpoint `/graphql` con un tipo `Model` cuyo campo `metrics` se resuelve aparte.

In [ ]:
import strawberry
from typing import List
from fastapi import FastAPI, HTTPException
from strawberry.fastapi import GraphQLRouter

MODELS = [
    {"id":"m1","name":"churn","version":3,"framework":"sklearn",
     "description":"Predice abandono de clientes; entrenado sobre transacciones_v3. " * 4,
     "params":{"n_estimators":200,"max_depth":8},
     "metrics":{"auc":0.87,"accuracy":0.81,"f1":0.79}},
    {"id":"m2","name":"fraude","version":1,"framework":"xgboost",
     "description":"Detección de fraude en pagos en tiempo real. " * 4,
     "params":{"eta":0.1,"max_depth":6},
     "metrics":{"auc":0.93,"accuracy":0.90,"f1":0.71}},
    {"id":"m3","name":"recomendador","version":2,"framework":"lightfm",
     "description":"Recomendación de productos por filtrado colaborativo. " * 4,
     "params":{"components":64},
     "metrics":{"auc":0.78,"accuracy":0.74,"f1":0.70}},
]
def _by_id(mid):
    return next((m for m in MODELS if m["id"] == mid), None)

app = FastAPI(title="REST vs GraphQL")

# ---------- REST ----------
@app.get("/rest/models")
def rest_models():
    # Objeto (casi) completo, SIN metrics -> para el dashboard hará falta otra llamada por modelo
    return [{k: v for k, v in m.items() if k != "metrics"} for m in MODELS]

@app.get("/rest/models/{mid}/metrics")
def rest_metrics(mid: str):
    m = _by_id(mid)
    if not m:
        raise HTTPException(status_code=404, detail="modelo no encontrado")
    return m["metrics"]

# ---------- GraphQL ----------
@strawberry.type
class Metrics:
    auc: float
    accuracy: float
    f1: float

@strawberry.type
class Model:
    id: str
    name: str
    version: int

    @strawberry.field
    def metrics(self) -> Metrics:
        mm = _by_id(self.id)["metrics"]
        return Metrics(auc=mm["auc"], accuracy=mm["accuracy"], f1=mm["f1"])

@strawberry.type
class Query:
    @strawberry.field
    def models(self) -> List[Model]:
        return [Model(id=m["id"], name=m["name"], version=m["version"]) for m in MODELS]

schema = strawberry.Schema(query=Query)
app.include_router(GraphQLRouter(schema), prefix="/graphql")
print("App lista: REST en /rest/... y GraphQL en /graphql")

## 3. Levantar la app en segundo plano

Para poder consultarla desde el mismo notebook, la corremos en un **hilo aparte** (no bloquea el kernel). GraphiQL queda en <http://localhost:8000/graphql>.

In [ ]:
import threading, time, uvicorn

def _run():
    uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning")).run()

threading.Thread(target=_run, daemon=True).start()
time.sleep(2)  # dar tiempo a que levante
BASE = "http://127.0.0.1:8000"
print("Servidor corriendo en", BASE)

## 4. Con REST: armar el dashboard (nombre + AUC de todos)

`GET /rest/models` **no** trae las métricas, así que necesitamos **una llamada más por modelo** para obtener el AUC. Contamos llamadas y bytes.

In [ ]:
import requests

llamadas_rest = 0
bytes_rest = 0

r = requests.get(f"{BASE}/rest/models"); llamadas_rest += 1; bytes_rest += len(r.content)
dashboard_rest = []
for m in r.json():
    rm = requests.get(f"{BASE}/rest/models/{m['id']}/metrics"); llamadas_rest += 1; bytes_rest += len(rm.content)
    dashboard_rest.append((m["name"], rm.json()["auc"]))

print("Dashboard:", dashboard_rest)
print(f"REST -> {llamadas_rest} llamadas, {bytes_rest} bytes transferidos")
print("Nota: /rest/models trajo description y params que NO necesitábamos (over-fetching).")

Observa dos cosas:
* **N+1:** 1 llamada para la lista + 1 por cada modelo = *n+1* llamadas.
* **Over-fetching:** `/rest/models` devolvió `description`, `params`, `framework`… aunque solo queríamos `name`.

## 5. Con GraphQL: una sola consulta, exactamente lo pedido

Pedimos `name` y `metrics { auc }` de todos los modelos en **una** query.

In [ ]:
query = """
query {
  models {
    name
    metrics { auc }
  }
}
"""
rg = requests.post(f"{BASE}/graphql", json={"query": query})
bytes_gql = len(rg.content)
dashboard_gql = [(x["name"], x["metrics"]["auc"]) for x in rg.json()["data"]["models"]]

print("Dashboard:", dashboard_gql)
print(f"GraphQL -> 1 llamada, {bytes_gql} bytes transferidos")

## 6. La comparación, en números

In [ ]:
print(f"{'':12}{'llamadas':>10}{'bytes':>10}")
print(f"{'REST':12}{llamadas_rest:>10}{bytes_rest:>10}")
print(f"{'GraphQL':12}{1:>10}{bytes_gql:>10}")
print()
print(f"GraphQL usó {llamadas_rest}x menos llamadas y ~{bytes_rest/max(bytes_gql,1):.1f}x menos datos para el MISMO resultado.")

## 7. Explorar en GraphiQL

Abre <http://localhost:8000/graphql> y prueba pedir solo `name`, o agregar `metrics { auc accuracy f1 }`. La **misma** API responde exactamente lo que pidas.

## Conclusión

| Aspecto | REST | GraphQL |
|---|---|---|
| Llamadas para la vista | n + 1 | 1 |
| Datos de más (over-fetching) | Sí | No |
| Forma de la respuesta | Fija por endpoint | La define el cliente |
| Caching HTTP | Simple y maduro | Más complejo |

**Regla práctica:** REST sigue siendo ideal para el borde simple y cacheable; GraphQL brilla cuando el cliente necesita **flexibilidad** o **agregar** datos relacionados. No es "uno u otro": conviven.

> **Mini-TP 2:** haz esta misma comparación con **tu** modelo — sírvelo por REST (Sesión 1) y por GraphQL, y reporta llamadas y datos para armar una vista de sus métricas.